# Fig 2b–c and Supp Fig 2 marker dot plots

Produces (`figures/paper/fig2bc_suppfig2_marker_dotplots/`, and the panels folder):
- **Fig 2b**: level-4 malignant states (`dotplot_celltype_level1_custom_Malignant`).
- **Fig 2c**: level-4 immune (myeloid + lymphoid) states (`dotplot_celltype_level1_custom_Immune`).
  The legend says "myeloid"; the published panel and the text (lymphoid cells cited as Fig 2c)
  are this immune plot.
- **Supp Fig 2** dot plots: `dotplot_celltype_level1_custom_Stromal` and the per-lineage
  `dotplot_celltype_level1_curated_<lineage>` plots (all markers). The Supp Fig 2 legend was not
  available; the panel ids (2c malignant, 2f immune, 2i glia / neuron / vascular) follow the text.

The dendrogram and colour squares of Fig 2b–c were added in the figure layout.

Origin: `analysis/annotation/marker_dotplots.ipynb`. Published files: EFS `figures/suppl_figures2/`
(also `figures/scRNAseq/annotation/`).

Comparison: the 9 dot-plot tables are identical to the published ones (EFS
`figures/scRNAseq/annotation/`: same rows and order, max |difference| 0). The dot plots use
`adata.raw` (counts), as `sc.pl.dotplot` does by default when `.raw` is present.

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

from gbmcollaboration import AllDataCenters
from gbmcollaboration.paper import config, output, snrna

def get_dotplot_df(dp):
    mean_long = dp.dot_color_df.reset_index().melt(
        id_vars=dp.dot_color_df.index.name or "index",
        var_name="celltype",
        value_name="mean_expression"
    )

    frac_long = dp.dot_size_df.reset_index().melt(
        id_vars=dp.dot_size_df.index.name or "index",
        var_name="celltype",
        value_name="fraction_expressing"
    )

    df = mean_long.merge(
        frac_long,
        on=[mean_long.columns[0], "celltype"]
    )
    return df

# Parameters

In [ ]:
OUT = output.setup("fig2bc_suppfig2_marker_dotplots")
CELL_TYPE_INFO_XLSX = config.GLIMMER_MARKERS_XLSX

# Manuscript panel of each dot plot (published files: EFS figures/suppl_figures2/)
PANELS = {
    'dotplot_celltype_level1_custom_Malignant': 'Fig 2b',
    'dotplot_celltype_level1_custom_Immune': 'Fig 2c',
    'dotplot_celltype_level1_custom_Stromal': 'Supp Fig 2i Stromal',
    'dotplot_celltype_level1_curated_Malignant': 'Supp Fig 2c Malignant',
    'dotplot_celltype_level1_curated_Myeloid': 'Supp Fig 2f Myeloid',
    'dotplot_celltype_level1_curated_Lymphoid': 'Supp Fig 2f Lymphoid',
    'dotplot_celltype_level1_curated_Glia': 'Supp Fig 2i Glia',
    'dotplot_celltype_level1_curated_Neuron': 'Supp Fig 2i Neuron',
    'dotplot_celltype_level1_curated_Vascular': 'Supp Fig 2i Vascular',
}

# Load data

In [ ]:
# load sc data
ad = AllDataCenters.load_singlecell(
    exclude_idh1_mut = True,
    exclude_patient_ids = True,
    exclude_relapse = True
)

# ensure raw counts are in .X. clear log1p status if present
if 'log1p' in ad.uns:
    del ad.uns['log1p']
ad.X = ad.layers['unnormalized_counts']
sc.pp.normalize_total(ad)
sc.pp.log1p(ad)

# load cell type name mapping
df_markers, df_ct_map = pd.read_excel(CELL_TYPE_INFO_XLSX,sheet_name=None).values()
df_markers['Level 4 Markers (WIP)'] = df_markers['Level 4 Markers (WIP)'].astype(str)
markers_map = df_markers[['celltype_level4_curated','Level 4 Markers (WIP)']].dropna().set_index('celltype_level4_curated').squeeze()
markers_map = markers_map.apply(lambda x: [s.strip() for s in x.split(',')]).to_dict()

# create columns with simplified cell type names
snrna.add_curated_levels(ad, df_ct_map.set_index('celltype_level4_scanvi'))

ct_groups_lvl1_custom = {
    'Malignant': 'Malignant',
    'Other': 'Other',
    'Neuron': 'Stromal',
    'Glia': 'Stromal',
    'Vascular': 'Stromal',
    'Lymphoid': 'Immune',
    'Myeloid': 'Immune'
}
ad.obs['celltype_level1_custom'] = ad.obs['celltype_level1_curated'].replace(ct_groups_lvl1_custom)

# exclude cell types
exclude_celltypes = [
    'Perivascular_fibroblast_cycling',
    'cDC2_cycling'
]

ad = ad[~ad.obs['celltype_level4_curated'].isin(exclude_celltypes)].copy()
assert ad.raw is not None  # sc.pl.dotplot plots .raw (counts) when present

In [ ]:
celltype_levels = [
    'celltype_level1_custom',
    'celltype_level1_curated',
    # 'celltype_level2_curated',
    # 'celltype_level3_curated',
]

cell_types_custom_order = [
        "Neuron_nos", "Neuron_excitatory_nos", "Neuron_excitatory_CDH24",
        "Neuron_inhibitory_LAMP5", "Neuron_inhibitory_PVALB", "Neuron_inhibitory_SST",
        "Neuron_inhibitory_VIP", "Neuron_inhibitory_NGF", "Astrocyte_homeostatic",
        "Astrocyte_reactive", "Astrocyte_inflammatory", "Astrocyte_IFN_response", "Astrocyte_intermediate_reactive",
        "Oligodendrocyte_IFN_response", "Oligodendrocyte_nos", "Oligodendrocyte_reactive",
        "Endothelial_arterial", "Endothelial_capillary", "Endothelial_veinous",
        "Endothelial_tip", "Endothelial_cycling", "Endothelial", "Pericyte",
        "Pericyte_cycling", "Pericyte_fibro-like", "Perivascular_fibroblast_type-I",
        "Perivascular_fibroblast_type-II", "Perivascular_fibroblast_type-III", "VSMC",
        #
        "Malignant_AC1", "Malignant_AC2", "Malignant_AC-like_IFN_response",
        "Malignant_MES1", "Malignant_MES-hypoxia", "Malignant_OPC1",
        "Malignant_OPC2", "Malignant_OPC-like_invasive", "Malignant_NPC2",
        "Malignant_cycling", "Malignant_other_1", "Malignant_other_2",
        "Malignant_other_stressed",
        #
        "Macrophage_C1Q_pos", "Macrophage_cycling", "Macrophage_IFN_response",
        "Macrophage_PVM", "Macrophage_LAM", "Monocyte_nos", "Microglia_homeostatic",
        "Microglia_intermediate_reactive", "Microglia_IFN_response",
        "Microglia_TREM2_pos", "cDC1", "cDC2", "cDC2_cycling", "cDC_CD207_pos",
        "pDC", "Neutrophil", "Neutrophil_immature", "Mast_cell", "T_cell_CD4",
        "T_cell_CD8", "T_reg", "T_cell_IFN_response", "T_NK_cycling",
        "T_cell_nos", "T_cell_other", "NK_nos", "NK_KLRC1_pos", "B_cell",
        "Plasma_cell", "Plasmablast"
    ]

for level in celltype_levels:

    groups = np.unique(ad.obs[level])
    
    for group_name in groups:
        print(f"Processing {level} {group_name}")

        if any(s in group_name for s in ['Doublet','Other']):
            continue

        # subset AnnData to just this group
        ad_sub = ad[ad.obs[level] == group_name].copy()
        ad_sub.obs['celltype_level4_curated'] = ad_sub.obs['celltype_level4_curated'].astype(str)

        u_ct = np.unique(ad_sub.obs['celltype_level4_curated'])

        if level == 'celltype_level1_custom':
            u_ct_ordered = [ct for ct in cell_types_custom_order if ct in u_ct]
            assert set(u_ct) == set(u_ct_ordered)
            max_n_markers = 2
        else:
            u_ct_ordered = u_ct
            max_n_markers = 100


        markers_map_grp = {}
        for ct in u_ct_ordered:
            ct_gene_list = markers_map[ct]
            markers_map_grp[ct] = []
            for g in ct_gene_list:
                g_clean = re.sub(r'\s*\(.*\)', '', g).strip()
                if g_clean in ad.var_names:
                    markers_map_grp[ct].append(g_clean)
                else:
                    print("\t",ct,"missing gene:", g_clean)


        genes_to_use = [g for genes in markers_map_grp.values() for g in genes[:max_n_markers]]
        unique_genes_to_use = list(dict.fromkeys(genes_to_use))

        mask = np.isin(ad_sub.var_names, unique_genes_to_use)

        if not len(unique_genes_to_use):
            print("Skipping", group_name)
            continue

        # rank genes for this group
        # sc.tl.rank_genes_groups(ad_sub, groupby=level, mask_var=mask)

        # create dotplot (or any plot)
        # fig = sc.pl.rank_genes_groups_dotplot(ad_sub, standard_scale='var', var_names=markers_map_grp,return_fig=True)

        dp = sc.pl.dotplot(
            ad_sub,
            var_names=unique_genes_to_use,
            groupby='celltype_level4_curated', 
            categories_order=u_ct_ordered,   
            standard_scale='var', 
            use_raw=True,
            return_fig=True,
            swap_axes=True,
            show=False,
            
        )
        # dp = dp.style(largest_dot=100)
        dp.make_figure()
        
        # Move x-axis ticks to the top
        ax = dp.get_axes()['mainplot_ax']
        ax.xaxis.tick_top()
        ax.xaxis.set_label_position('top')
        ax.tick_params(
            axis='x',
            top=True,bottom=False,
            labeltop=True,labelbottom=False,
            # pad=150 
        )
        # for label in ax.get_xticklabels():
        #     label.set_verticalalignment('top') 
        fig = dp.fig
        current_w, current_h = fig.get_size_inches()
        fig.set_size_inches(current_w * .8, current_h * .8)

        # get df
        df_plot = get_dotplot_df(dp)

        # save plot
        name = f'dotplot_{level}_{group_name}'
        output.save(OUT, name, fig=fig, data=df_plot, panel=PANELS.get(name))
        plt.close(fig)

        # delete subset to free memory
        del ad_sub